# Credential Stuffing Simulator — Demo Notebook

**Student:** Jejo J  
**Roll No:** 727823tucy019  
**Project:** Credential Stuffing Simulator  
**Date:** 2026-03-29  

---

## What is Credential Stuffing?

Credential stuffing is an automated attack where leaked username/password pairs
are "stuffed" into login forms of other services. Attackers rely on users
reusing passwords across sites.

**This notebook simulates the detection side** — how a server would respond to
such attempts using rate-limiting and account locking.

> ⚠️ For academic demonstration only. No real systems are involved.

In [ ]:
# Setup: add project root to path
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from code.helper_modules.auth_simulator import AuthSimulator, USER_DB
from datetime import datetime

print(f'Roll No: 727823tucy019 | Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Dummy user database has {len(USER_DB)} users: {list(USER_DB.keys())}')

## Test Case 1 — Mixed Success + Failure

In [ ]:
sim = AuthSimulator(log_path='/dev/null')

tc1 = [
    ('alice',   'password123'),   # correct
    ('bob',     'wrongpass'),     # wrong
    ('charlie', 'ch4rlie_rocks'), # correct
    ('diana',   'badpassword'),   # wrong
    ('eve',     'ev3pass'),       # correct
    ('frank',   'wrong123'),      # wrong
]

print('TC-1 Results:')
print(f'{"Username":<12} {"Password":<20} {"Status"}')
print('-' * 48)
for user, pwd in tc1:
    status = sim.attempt_login(user, pwd)
    print(f'{user:<12} {pwd:<20} {status}')

## Test Case 2 — All Invalid Credentials

In [ ]:
sim2 = AuthSimulator(log_path='/dev/null')

tc2 = [
    ('alice',   'notherpassword'),
    ('bob',     '12345'),
    ('charlie', 'wrongwrongwrong'),
    ('',        'somepass'),       # empty username
    ('frank',   ''),              # empty password
    ('unknown', 'doesntexist'),   # user not in DB
]

print('TC-2 Results:')
print(f'{"Username":<12} {"Password":<20} {"Status"}')
print('-' * 48)
for user, pwd in tc2:
    status = sim2.attempt_login(user, pwd)
    print(f'{repr(user):<12} {repr(pwd):<20} {status}')

## Test Case 3 — Account Lock Scenario

In [ ]:
sim3 = AuthSimulator(log_path='/dev/null')

tc3 = [
    ('alice', 'wrong1'),      # fail 1
    ('alice', 'wrong2'),      # fail 2
    ('alice', 'wrong3'),      # fail 3 → LOCKS
    ('alice', 'password123'), # correct but LOCKED
    ('bob',   'wrong1'),      # bob fail 1
    ('bob',   'securepass!'), # bob succeeds
]

print('TC-3 Results:')
print(f'{"Username":<12} {"Password":<20} {"Status"}')
print('-' * 48)
for user, pwd in tc3:
    status = sim3.attempt_login(user, pwd)
    print(f'{user:<12} {pwd:<20} {status}')

## Summary

| Status | Meaning |
|--------|---------|
| SUCCESS | Correct username + password |
| FAILED  | Wrong password (account not yet locked) |
| LOCKED  | Account locked after 3 failures |
| INVALID FORMAT | Empty or malformed username/password |

Account locking is a standard defense mechanism against credential stuffing —
it limits the number of attempts an attacker can make per account.